# Практическое задание №4. Обучение с учителем. Линейные модели.

## Установка зависимостей

In [ ]:
%pip install -qqq pyarrow fastparquet
%pip install scikit-learn
%pip install xgboost lightgbm catboost

## Импорт зависимостей

In [3]:
import numpy as np
import pickle
import pandas as pd 
from joblib import Parallel, delayed
from sklearn.base import BaseEstimator
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils import check_X_y, check_array, resample
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import BaggingRegressor, BaggingClassifier, RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier
from collections import Counter

## NYC Taxi Dataset


### Загружаем датасет и таблицу для геоданных

In [4]:
from pathlib import Path 

dataframes = list()

for file_name in Path("/home/jupyter/datasets/nyc-yellow-2025").iterdir():
    if file_name.suffix == '.parquet':
        dataframes.append(pd.read_parquet(file_name))
#print(dataframes) 

df = pd.concat(dataframes).drop(columns=['total_amount', 'tolls_amount', 'tip_amount', 'extra'])

zone_lookup = pd.read_csv('/home/jupyter/datasets/nyc-yellow-2025/taxi_zone_lookup.csv', index_col='LocationID')

#df

In [5]:
# Второе полугодие

dataframes_new = list()
dataframes_new.append(pd.read_parquet("nyc_2025_h2.parquet"))
df_new = pd.concat(dataframes).drop(columns=['total_amount', 'tolls_amount', 'tip_amount', 'extra'])
#df_new

### Преобразуем данные

In [6]:
with open('nyc_pipeline_old.pickle', 'rb') as f:
    nyc_pipeline = pickle.load(f)
target_NYC = df['fare_amount'].copy()
features_NYC = df.drop('fare_amount', axis=1)

target_NYC_new = df_new['fare_amount'].copy()
features_NYC_new = df_new.drop('fare_amount', axis=1)

features_train_NYC, features_test_NYC, target_train_NYC, target_test_NYC = train_test_split(
    features_NYC, target_NYC, test_size=0.2, random_state=44
)


X_train_NYC = nyc_pipeline.fit_transform(features_train_NYC).astype(np.float32)
X_test_NYC = nyc_pipeline.transform(features_test_NYC).astype(np.float32)

X_test_NYC_new = nyc_pipeline.transform(features_NYC_new).astype(np.float32)

## UK Car Accidents

### Загружаем данные

In [7]:
accidents = pd.read_csv(
    filepath_or_buffer='/home/jupyter/datasets/road-accidents/road-accident-united-kingdom-uk-dataset/UK_Accident.csv',
).drop(columns=['Accident_Index', 'Unnamed: 0', 'LSOA_of_Accident_Location'])

#accidents

### Преобразуем данные

In [8]:
with open('uk_pipeline_old.pickle', 'rb') as f:
    uk_pipeline = joblib.load(f)

target_UK = accidents['Accident_Severity']
target_UK = target_UK.apply(lambda x: 1 if x >= 2 else 0)  # бинаризация: серьёзное ДТП = 1
features_UK = accidents.drop(['Accident_Severity'], axis=1)

features_train_UK, features_test_UK, target_train_UK, target_test_UK = train_test_split(
    features_UK, target_UK, test_size=0.2, random_state=44
)

X_train_UK = uk_pipeline.fit_transform(features_train_UK)
X_test_UK = uk_pipeline.transform(features_test_UK)


# Второе задание

## Собственноручная реализация алгоритмов 

### Беггинг

In [9]:
class BaggingCustom(BaseEstimator):
    def __init__(self, regression=True, n_estimators=10, max_depth=None, 
                 random_state=None, n_jobs=None):
        self.regression = regression
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.n_jobs = n_jobs

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.n_features_in_ = X.shape[1]
        
        if not self.regression:
            self.classes_ = np.unique(y)
            tree_class = DecisionTreeClassifier
        else:
            tree_class = DecisionTreeRegressor

        rng = np.random.RandomState(self.random_state)

        def _fit_single_tree(_):
            X_res, y_res = resample(X, y, random_state=rng)
            tree = tree_class(max_depth=self.max_depth, random_state=rng.randint(0, int(1e9)))
            tree.fit(X_res, y_res)
            return tree

        self.estimators_ = Parallel(n_jobs=self.n_jobs)(
            delayed(_fit_single_tree)(None) for _ in range(self.n_estimators)
        )
        return self

    def predict(self, X):
        X = check_array(X)
        predictions = np.array([tree.predict(X) for tree in self.estimators_])
        
        if self.regression:
            return np.mean(predictions, axis=0)
        else:
            return np.array([Counter(preds).most_common(1)[0][0] 
                           for preds in predictions.T])

### RandomForest

In [10]:
class RandomForestCustom(BaseEstimator):
    def __init__(self, regression=True, n_estimators=10, max_depth=None, 
                 max_features='sqrt', random_state=None, n_jobs=None):
        self.regression = regression
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.n_jobs = n_jobs

    def _get_n_features(self, n_features):
        if self.max_features == 'sqrt':
            return int(np.sqrt(n_features))
        elif isinstance(self.max_features, int):
            return min(self.max_features, n_features)
        elif isinstance(self.max_features, float):
            return int(self.max_features * n_features)
        else:
            return n_features

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.n_features_in_ = X.shape[1]
        
        if not self.regression:
            self.classes_ = np.unique(y)
            tree_class = DecisionTreeClassifier
        else:
            tree_class = DecisionTreeRegressor

        n_samples, n_features = X.shape
        rng = np.random.default_rng(self.random_state)
        n_sub_features = self._get_n_features(n_features)

        def _fit_single_tree(_):
            bootstrap_idx = rng.choice(n_samples, size=n_samples, replace=True)
            feature_idx = rng.choice(n_features, size=n_sub_features, replace=False)
            tree = tree_class(max_depth=self.max_depth, random_state=rng.integers(1e9))
            tree.fit(X[bootstrap_idx][:, feature_idx], y[bootstrap_idx])
            return tree, feature_idx

        self.estimators_ = Parallel(n_jobs=self.n_jobs)(
            delayed(_fit_single_tree)(None) for _ in range(self.n_estimators)
        )
        return self

    def predict(self, X):
        X = check_array(X)
        predictions = np.array([
            tree.predict(X[:, features]) for tree, features in self.estimators_
        ])
        
        if self.regression:
            return np.mean(predictions, axis=0)
        else:
            return np.array([Counter(preds).most_common(1)[0][0] 
                           for preds in predictions.T])

## Сравнение:

In [11]:
def compare_models(custom_model, sklearn_model, X_train, y_train, X_test, y_test, 
                   task="regression", model_name="Model"):
    print(f"Сравнение: {model_name}")
    
    # Обучение и предсказание
    custom_pred = custom_model.fit(X_train, y_train).predict(X_test)
    sklearn_pred = sklearn_model.fit(X_train, y_train).predict(X_test)

    if task == "regression":
        for name, pred in [("Кастомная модель:", custom_pred), ("sklearn модель:", sklearn_pred)]:
            print(f"\n{name}")
            print(f"  MSE: {mean_squared_error(y_test, pred)}")
            print(f"  R2:  {r2_score(y_test, pred)}")
    else:  # классификация
        for name, pred in [("Кастомная модель:", custom_pred), ("sklearn модель:", sklearn_pred)]:
            print(f"\n{name}")
            print(f"  Accuracy:  {accuracy_score(y_test, pred)}")
            print(f"  Precision: {precision_score(y_test, pred, average='binary', zero_division=0)}")
            print(f"  Recall:    {recall_score(y_test, pred, average='binary', zero_division=0)}")
            print(f"  F1-score:  {f1_score(y_test, pred, average='binary', zero_division=0)}")

### Линейная регрессия 

In [20]:
# Для регрессии
X_NYC_small, y_NYC_small = resample(
    X_train_NYC, target_train_NYC,
    n_samples=300000,
    random_state=42
)

X_NYC_small_train, X_NYC_small_test, y_NYC_small_train, y_NYC_small_test = train_test_split(
    X_NYC_small, y_NYC_small, test_size=0.2, random_state=42
)


X_NYC_small_new, y_NYC_small_new = resample(
    X_test_NYC_new, target_NYC_new,
    n_samples=300000,
    random_state=42
)


# Для классификации
X_UK_small, y_UK_small = resample(
    X_train_UK, target_train_UK,  
    n_samples=100000,
    random_state=42
)

X_UK_small_train, X_UK_small_test, y_UK_small_train, y_UK_small_test = train_test_split(
    X_UK_small, y_UK_small, test_size=0.2, random_state=42
)
print(y_NYC_small.mean())

17.992534933333335


In [13]:
rf_custom_reg = RandomForestCustom(regression=True, n_estimators=40, max_depth=None, random_state=42, n_jobs=4)
rf_sklearn_reg = RandomForestRegressor(n_estimators=40, max_depth=None, random_state=42, n_jobs=4)

compare_models(
    rf_custom_reg, rf_sklearn_reg,
    X_NYC_small_train, y_NYC_small_train, X_NYC_small_test, y_NYC_small_test,
    task="regression", model_name="Random Forest (Regression)"
)

rf_custom_clf = RandomForestCustom(regression=False, n_estimators=30, max_depth=3, random_state=42, n_jobs=6)
rf_sklearn_clf = RandomForestClassifier(n_estimators=30, max_depth=3, random_state=42, n_jobs=6)

compare_models(
    rf_custom_clf, rf_sklearn_clf,
    X_UK_small_train, y_UK_small_train, X_UK_small_test, y_UK_small_test,
    task="classification", model_name="Random Forest (Classification)"
)


Сравнение: Random Forest (Regression)

Кастомная модель:
  MSE: 315.33730403252576
  R2:  0.08481010909038988

sklearn модель:
  MSE: 3209.766500480471
  R2:  -8.31556722231965
Сравнение: Random Forest (Classification)

Кастомная модель:
  Accuracy:  0.98725
  Precision: 0.98725
  Recall:    1.0
  F1-score:  0.9935840986287583

sklearn модель:
  Accuracy:  0.98725
  Precision: 0.98725
  Recall:    1.0
  F1-score:  0.9935840986287583


### Беггинг

In [14]:
bag_custom_reg = BaggingCustom(regression=True, n_estimators=40, max_depth=None, random_state=42, n_jobs=4)
bag_sklearn_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=None),
    n_estimators=100,
    random_state=42,
    n_jobs=4
)

compare_models(
    bag_custom_reg, bag_sklearn_reg,
    X_NYC_small_train, y_NYC_small_train, X_NYC_small_test, y_NYC_small_test,
    task="regression", model_name="Bagging (Regression)"
)

bag_custom_clf = BaggingCustom(regression=False, n_estimators=30, max_depth=3, random_state=42, n_jobs=6)
bag_sklearn_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=3),
    n_estimators=30,
    random_state=42,
    n_jobs=6
)

compare_models(
    bag_custom_clf, bag_sklearn_clf,
    X_UK_small_train, y_UK_small_train, X_UK_small_test, y_UK_small_test,
    task="classification", model_name="Bagging (Classification)"
)

Сравнение: Bagging (Regression)

Кастомная модель:
  MSE: 285.79396610333333
  R2:  0.17055246773545052

sklearn модель:
  MSE: 1753.3354744554301
  R2:  -4.088630114720911
Сравнение: Bagging (Classification)

Кастомная модель:
  Accuracy:  0.9873
  Precision: 0.9872993649682484
  Recall:    1.0
  F1-score:  0.9936090982286635

sklearn модель:
  Accuracy:  0.98725
  Precision: 0.98725
  Recall:    1.0
  F1-score:  0.9935840986287583


## Беггинг и случайный лес VS Градиентный бустинг

In [15]:
# Для сравнения
def evaluate_model(model, X_train, y_train, X_test, y_test, task="regression", model_name="Model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if task == "regression":
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        print(f"{model_name} | MSE: {mse} | R2: {r2}")
        return {"mse": mse, "r2": r2}
    
    else:  # классификация
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='binary', zero_division=0)
        rec = recall_score(y_test, y_pred, average='binary', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='binary', zero_division=0)
        
        print(f"{model_name} | Acc: {acc} | Prec: {prec} | Rec: {rec} | F1: {f1}")
        return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

In [16]:
print("РЕГРЕССИЯ (NYC Taxi)")
print("Модель")

N_ESTIMATORS = 40
MAX_DEPTH = None
RANDOM_STATE = 42

models_reg = [
    ("Bagging (sklearn)", BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=MAX_DEPTH, random_state=RANDOM_STATE),
        n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=6)),
    
    ("Random Forest (sklearn)", RandomForestRegressor(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE, n_jobs=6)),
    
    ("GB (sklearn)", GradientBoostingRegressor(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE)),
    
    ("XGBoost", XGBRegressor(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE, 
        n_jobs=4, verbosity=0)),
    
    ("LightGBM", LGBMRegressor(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE, 
        n_jobs=4, verbose=-1)),
    
    ("CatBoost", CatBoostRegressor(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE,
        verbose=0, thread_count=4))
]

results_reg = {}
for name, model in models_reg:
    try:
        results_reg[name] = evaluate_model(model, X_NYC_small_train, y_NYC_small_train, X_NYC_small_test, y_NYC_small_test, 
                                           task="regression", model_name=name)
    except Exception as e:
        print(f"{name} | ошибка {str(e)}")

РЕГРЕССИЯ (NYC Taxi)
Модель
Bagging (sklearn) | MSE: 3811.909720038354 | R2: -10.063141582764805
Random Forest (sklearn) | MSE: 3209.76650048047 | R2: -8.315567222319649
GB (sklearn) | MSE: 214.36341432846618 | R2: 0.37786228503417096
XGBoost | MSE: 124.42631138681556 | R2: 0.6388828695870398
LightGBM | MSE: 756.0598293770048 | R2: -1.1942799152532513
CatBoost | MSE: 2181.528507818009 | R2: -5.33135633353495


In [17]:
print("КЛАССИФИКАЦИЯ (UK)")
print("Модель")

models_clf = [
    ("Bagging (sklearn)", BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
        n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=6)),
    
    ("Random Forest (sklearn)", RandomForestClassifier(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE, n_jobs=6)),
    
    ("GB (sklearn)", GradientBoostingClassifier(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE)),
    
    ("XGBoost", XGBClassifier(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE,
        n_jobs=4, verbosity=0)),
    
    ("LightGBM", LGBMClassifier(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE,
        n_jobs=4, verbose=-1)),
    
    ("CatBoost", CatBoostClassifier(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE,
        verbose=0, thread_count=4))
]

results_clf = {}
for name, model in models_clf:
    try:
        results_clf[name] = evaluate_model(model, X_UK_small_train, y_UK_small_train, X_UK_small_test, y_UK_small_test,
                                           task="classification", model_name=name)
    except Exception as e:
        print(f"{name} | ошибка {str(e)}")

КЛАССИФИКАЦИЯ (UK)
Модель
Bagging (sklearn) | Acc: 0.98725 | Prec: 0.98725 | Rec: 1.0 | F1: 0.9935840986287583
Random Forest (sklearn) | Acc: 0.9884 | Prec: 0.9883866446413375 | Rec: 1.0 | F1: 0.9941594078847994
GB (sklearn) | Acc: 0.97515 | Prec: 0.988577520560463 | Rec: 0.9862243605976196 | F1: 0.987399538574652
XGBoost | Acc: 0.9873 | Prec: 0.9872993649682484 | Rec: 1.0 | F1: 0.9936090982286635
LightGBM | Acc: 0.9872 | Prec: 0.9872493624681234 | Rec: 0.999949354266903 | F1: 0.9935587761674719
CatBoost | Acc: 0.9872 | Prec: 0.9872493624681234 | Rec: 0.999949354266903 | F1: 0.9935587761674719


### Данные за второе полугодие

In [21]:
compare_models(
    rf_custom_reg, rf_sklearn_reg,
    X_NYC_small_train, y_NYC_small_train, X_NYC_small_new, y_NYC_small_new,
    task="regression", model_name="Random Forest (Regression)"
)

compare_models(
    bag_custom_reg, bag_sklearn_reg,
    X_NYC_small_train, y_NYC_small_train, X_NYC_small_new, y_NYC_small_new,
    task="regression", model_name="Bagging (Regression)"
)



Сравнение: Random Forest (Regression)

Кастомная модель:
  MSE: 326.14646769696697
  R2:  0.08296652530920745

sklearn модель:
  MSE: 5215.184350989039
  R2:  -13.663652990974887
Сравнение: Bagging (Regression)

Кастомная модель:
  MSE: 428.23126514348144
  R2:  -0.20406763200217504

sklearn модель:
  MSE: 5117.285505131326
  R2:  -13.388388569382162


# Замечание

Мы заметили, что таргет очень странный: fare_amount - просто цена поездки (без налогов и т.д.)

In [22]:
print("y:  min={:.2f}, max={:.2f}, mean={:.2f}".format(target_NYC.min(), target_NYC.max(), target_NYC.mean()))
print("y_new:  min={:.2f}, max={:.2f}, mean={:.2f}".format(target_NYC_new.min(), target_NYC_new.max(), target_NYC_new.mean()))

y:  min=-1807.60, max=863372.12, mean=17.89
y_new:  min=-1807.60, max=863372.12, mean=17.89


Скорее всего, этот показатель измеряется в долларах

Цена поездки в -1000 долларов и в 863 тысячи долларов выглядят очень странно, поэтому нам захотелось посмотреть, как будут работать модели если обрезать таргет

In [24]:
def cut_target(X, y, min_fare=0, max_fare=1000):
    mask = (y >= min_fare) & (y <= max_fare)
    return X[mask], y[mask]

X_train_NYC, target_train_NYC = cut_target(X_train_NYC, target_train_NYC)
X_test_NYC, target_test_NYC = cut_target(X_test_NYC, target_test_NYC)
X_test_NYC_new, target_NYC_new = cut_target(X_NYC_small_new, y_NYC_small_new)

print("y_train: min={:.2f}, max={:.2f}, mean={:.2f}".format(target_test_NYC.min(), target_test_NYC.max(), target_test_NYC.mean()))
print("y_train2: min={:.2f}, max={:.2f}, mean={:.2f}".format(target_NYC_new.min(), target_NYC_new.max(), target_NYC_new.mean()))

y_train: min=0.00, max=990.00, mean=19.45
y_train2: min=0.00, max=900.00, mean=19.52


In [25]:
# Для регрессии
X_NYC_small, y_NYC_small = resample(
    X_train_NYC, target_train_NYC,
    n_samples=300000,
    random_state=42
)

X_NYC_small_train, X_NYC_small_test, y_NYC_small_train, y_NYC_small_test = train_test_split(
    X_NYC_small, y_NYC_small, test_size=0.2, random_state=42
)

print(y_NYC_small.mean())

19.440130066666672


In [26]:
rf_custom_reg = RandomForestCustom(regression=True, n_estimators=100, max_depth=None, random_state=42, n_jobs=4)
rf_sklearn_reg = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=4)

compare_models(
    rf_custom_reg, rf_sklearn_reg,
    X_NYC_small_train, y_NYC_small_train, X_NYC_small_test, y_NYC_small_test,
    task="regression", model_name="Random Forest (Regression)"
)

bag_custom_reg = BaggingCustom(regression=True, n_estimators=100, max_depth=None, random_state=42, n_jobs=4)
bag_sklearn_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=None),
    n_estimators=100,
    random_state=42,
    n_jobs=4
)

compare_models(
    bag_custom_reg, bag_sklearn_reg,
    X_NYC_small_train, y_NYC_small_train, X_NYC_small_test, y_NYC_small_test,
    task="regression", model_name="Bagging (Regression)"
)


Сравнение: Random Forest (Regression)

Кастомная модель:
  MSE: 209.98279515018322
  R2:  0.26749045073874744

sklearn модель:
  MSE: 58.93509482523049
  R2:  0.7944092528379647
Сравнение: Bagging (Regression)

Кастомная модель:
  MSE: 119.09208582333326
  R2:  0.5845560106739244

sklearn модель:
  MSE: 59.27913757427667
  R2:  0.7932090849915935


In [27]:
print("Сравнения")

print("РЕГРЕССИЯ (NYC Taxi)")
print("Модель")

N_ESTIMATORS = 40
MAX_DEPTH = None
RANDOM_STATE = 42

models_reg = [
    ("Bagging (sklearn)", BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=MAX_DEPTH, random_state=RANDOM_STATE),
        n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=4)),
    
    ("Random Forest (sklearn)", RandomForestRegressor(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE, n_jobs=4)),
    
    ("GB (sklearn)", GradientBoostingRegressor(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE)),
    
    ("XGBoost", XGBRegressor(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE, 
        n_jobs=4, verbosity=0)),
    
    ("LightGBM", LGBMRegressor(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE, 
        n_jobs=4, verbose=-1)),
    
    ("CatBoost", CatBoostRegressor(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE,
        verbose=0, thread_count=4))
]

results_reg = {}
for name, model in models_reg:
    try:
        results_reg[name] = evaluate_model(model, X_NYC_small_train, y_NYC_small_train, X_NYC_small_test, y_NYC_small_test, 
                                           task="regression", model_name=name)
    except Exception as e:
        print(f"{name} | ошибка {str(e)}")

Сравнения
РЕГРЕССИЯ (NYC Taxi)
Модель
Bagging (sklearn) | MSE: 59.16500475994793 | R2: 0.7936072289267672
Random Forest (sklearn) | MSE: 59.198331540814586 | R2: 0.7934909709008965
GB (sklearn) | MSE: 115.72919879021454 | R2: 0.5962871949506385
XGBoost | MSE: 54.51619606180843 | R2: 0.8098242564296176
LightGBM | MSE: 59.884938753617405 | R2: 0.7910957920977474
CatBoost | MSE: 56.726510583387025 | R2: 0.8021137366569423


## Второе полугодие

In [30]:
compare_models(
    rf_custom_reg, rf_sklearn_reg,
    X_NYC_small_train, y_NYC_small_train, X_test_NYC_new, target_NYC_new,
    task="regression", model_name="Random Forest (Regression)"
)


bag_custom_reg = BaggingCustom(regression=True, n_estimators=100, max_depth=None, random_state=42, n_jobs=2)
bag_sklearn_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=None),
    n_estimators=100,
    random_state=42,
    n_jobs=2
)

compare_models(
    bag_custom_reg, bag_sklearn_reg,
    X_NYC_small_train, y_NYC_small_train, X_test_NYC_new, target_NYC_new,
    task="regression", model_name="Bagging (Regression)"
)


Сравнение: Random Forest (Regression)

Кастомная модель:
  MSE: 220.66242144137766
  R2:  0.2644324131885988

sklearn модель:
  MSE: 58.171932921270226
  R2:  0.8060866547210391
Сравнение: Bagging (Regression)

Кастомная модель:
  MSE: 135.07083573053438
  R2:  0.5497478544922829

sklearn модель:
  MSE: 57.972168909613565
  R2:  0.8067525584622662


# Вывод

Классификация показывает отличные результаты

Регрессия показывает странные результаты

Возможные причины (по нашему мнению):

1. Мы используем не все данные, а сэмплы по 300к объектов
2. Наш target имеет огромные выбросы

Обрезав таргет от 0 до 1000 результаты работы моделей улучшаются

Таблица: https://docs.google.com/spreadsheets/d/1AFj8Bp8Xx9VKLjSN7OLMV35RKgE9WeslFf6kXcCTUS0/edit?gid=0#gid=0
